In [1]:
import numpy as np
from PIL import Image
import cv2
import tensorflow as tf
import time
import requests  # For sending HTTP alerts
from datetime import datetime
import json

In [2]:
from PytorchWildlife.models import detection as pw_detection

detection_model = pw_detection.MegaDetectorV6(
    pretrained=True,
    version="MDV6-yolov10-c" 
)

# Run detection
img_path2 = "C:/Users/yrosh/Downloads/zebra.jpg"
img_path = "C:/Users/yrosh/Downloads/zebra.jpg"
detection_result = detection_model.single_image_detection(img_path)
detection_result

Ultralytics 8.4.40  Python-3.12.10 torch-2.11.0+cpu CPU (11th Gen Intel Core i5-11320H @ 3.20GHz)
YOLOv10n summary (fused): 101 layers, 2,265,753 parameters, 0 gradients, 6.5 GFLOPs

0: 1280x1280 3 animals, 1306.0ms
Speed: 24.6ms preprocess, 1306.0ms inference, 12.7ms postprocess per image at shape (1, 3, 1280, 1280)


{'img_id': 'C:/Users/yrosh/Downloads/zebra.jpg',
 'detections': Detections(xyxy=array([[      402.8,      72.506,      1001.1,      667.88],
        [     664.22,      234.28,      1254.2,      844.21],
        [     544.39,      232.28,      1076.5,       841.7]], dtype=float32), mask=None, confidence=array([    0.67021,     0.62807,     0.22553], dtype=float32), class_id=array([0, 0, 0]), tracker_id=None, data={}),
 'labels': ['animal 0.67', 'animal 0.63', 'animal 0.23'],
 'normalized_coords': [[np.float32(0.31468597),
   np.float32(0.085001364),
   np.float32(0.7821322),
   np.float32(0.7829729)],
  [np.float32(0.518925),
   np.float32(0.27465445),
   np.float32(0.9798813),
   np.float32(0.9896987)],
  [np.float32(0.4253057),
   np.float32(0.27231494),
   np.float32(0.8410465),
   np.float32(0.9867549)]]}

In [3]:
# Load  trained model and class names
model = tf.keras.models.load_model('./wildlife_classifier_class_15_acc_93.keras')

with open('./class_labels_class_15_acc_93.json', 'r') as f:
    class_names = json.load(f)

class_names

['Impala',
 'Indian_Elephant',
 'brown_bear',
 'cheetah',
 'coyote_indian_jackal',
 'gazelle',
 'jaguar',
 'langur',
 'leopard',
 'lion',
 'macaque',
 'puma',
 'sloth_bear',
 'tiger',
 'timber_wolf']

In [4]:
def classify_image_cnn(cropped_img, confidence_threshold=0.3):
    
    if cropped_img is None or cropped_img.size == 0:
        return "Invalid_Image", 0.0
    
    try:
        # Preprocessing
        img = cv2.cvtColor(cropped_img, cv2.COLOR_BGR2RGB)
        img = cv2.resize(img, (160, 160))           
        img = img / 255.0
        img = np.expand_dims(img, axis=0)
        
        # Predict
        predictions = model.predict(img, verbose=0)
        class_idx = np.argmax(predictions[0])
        confidence = float(predictions[0][class_idx])
        
        species = class_names[class_idx]
        
        if confidence < confidence_threshold:
            return "Unknown", confidence
            
        return species, confidence
    
    except Exception as e:
        print(f"Classification error: {e}")
        return "Error", 0.0
    
    
image = cv2.imread("C:/Users/yrosh/Downloads/chita.jpg")
classify_image_cnn(image)

('cheetah', 0.6697331666946411)

In [5]:
def detect_and_classify(image_path, det_conf_threshold=0.5, cls_conf_threshold=0.3, show=True):
    
    start_time = time.time()

    #for images from internet
    # resp = requests.get(image_path)
    # img_array = np.asarray(bytearray(resp.content), dtype=np.uint8)
    # img = cv2.imdecode(img_array, cv2.IMREAD_COLOR)
    
    # Read image
    img = cv2.imread(image_path)
    if img is None:
        print("Error: Could not load image")
        return None
    
    # Run detection
    detections = detection_model.single_image_detection(img)
    
    result_img = img.copy()
    results = []
    
    for box, conf, class_id in zip(detections['detections'].xyxy, detections['detections'].confidence, detections['detections'].class_id):
        if(class_id != 0):
            continue
        if conf < det_conf_threshold:
            continue
            
        x_min, y_min, x_max, y_max = map(int, box)
        
        # Crop detected animal
        cropped = img[y_min:y_max, x_min:x_max]
        
        # Classify with your CNN
        species, species_conf = classify_image_cnn(cropped, cls_conf_threshold)
        
        results.append({
            'species': species,
            'det_conf': float(conf),
            'cls_conf': species_conf,
            'bbox': [x_min, y_min, x_max, y_max]
        })
        
        # Draw on image
        color = (0, 255, 0) if species != "Unknown" else (0, 165, 255)
        cv2.rectangle(result_img, (x_min, y_min), (x_max, y_max), color, 2)
        
        label = f"{species} {species_conf:.2f}"
        cv2.putText(result_img, label, (x_min, y_min - 8),
                    cv2.FONT_HERSHEY_SIMPLEX, 0.65, color, 2)
    
    total_time = time.time() - start_time
    
    # Print results
    print(f"\nDetection + Classification completed in {total_time:.2f} seconds")
    print(f"Detected {len(results)} animal(s)\n")
    
    for r in results:
        print(f"→ {r['species']:15} | Det_con: {r['det_conf']:.3f} | Classi_con: {r['cls_conf']:.3f}")
    
    # Show image
    if show:
        cv2.imshow("Wildlife Detection + Classification", result_img)
        cv2.waitKey(0)
        cv2.destroyAllWindows()
    
    
    return results


if __name__ == "__main__":
    
    # test_image = "./images/test5.jpg"   
    test_image = "./images/test4.jpg"   
    # test_image = "./images/test3.jpg"   
    # test_image = "./images/test2.jpg"   
    # test_image = "./images/test1.jpg"   
    detect_and_classify(test_image, det_conf_threshold=0.2, cls_conf_threshold=0.1)


0: 1280x1280 5 animals, 614.1ms
Speed: 18.5ms preprocess, 614.1ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

Detection + Classification completed in 1.32 seconds
Detected 5 animal(s)

→ tiger           | Det_con: 0.940 | Classi_con: 1.000
→ lion            | Det_con: 0.918 | Classi_con: 1.000
→ cheetah         | Det_con: 0.897 | Classi_con: 0.558
→ leopard         | Det_con: 0.786 | Classi_con: 0.810
→ jaguar          | Det_con: 0.745 | Classi_con: 0.709


KeyboardInterrupt: 

In [38]:
import time
import threading
import requests
import cv2
import base64
from collections import deque
from datetime import datetime


class EdgeAlertManager:
    def __init__(self, camera_id, zone, location):
        self.camera_id = camera_id
        self.zone = zone
        self.location = location
        
        # Buffer to keep track of species detected in the last 5 sampled frames
        self.frame_buffer = deque(maxlen=5) 
        
        # Dictionary to track when an alert was last sent for a species
        self.last_alert_times = {} 
        
        self.COOLDOWN_SECONDS = 300 # 5 minutes
        self.API_URL = "https://wildlife-81wn.onrender.com/api/alerts"

        # Define priority levels
        self.SEVERITY_MAP = {
            'leopard': 'CRITICAL',
            'tiger': 'CRITICAL',
            'sloth_bear': 'CRITICAL',
            'jaguar': 'CRITICAL',
            'cheetah': 'CRITICAL',
            'lion': 'HIGH',
            'puma': 'HIGH',
            'brown_bear': 'MEDIUM',
            'timber_wolf': 'MEDIUM',
            'coyote_indian_jackal': 'MEDIUM',
            'langur': 'LOW',
            'macaque': 'LOW',
        }

    # CHANGED: We now accept the full annotated_frame instead of best_crops_dict
    def process_frame_detections(self, current_frame_species, max_conf_dict, annotated_frame):
        self.frame_buffer.append(current_frame_species)
        
        # Flatten the buffer to count occurrences of each species in the last 5 frames
        species_counts = {}
        for frame_set in self.frame_buffer:
            for sp in frame_set:
                species_counts[sp] = species_counts.get(sp, 0) + 1
                
        # Only iterate over species detected in the CURRENT frame
        for species in current_frame_species:
            if species_counts.get(species, 0) >= 2: # Trigger threshold
                
                # Extract the highest confidence for this species
                conf = max_conf_dict[species]
                
                # Pass the full frame with bounding boxes to the alert function
                self._attempt_alert(species, conf, annotated_frame)

    def _attempt_alert(self, species, confidence, frame_image):
        current_time = time.time()
        last_time = self.last_alert_times.get(species, 0)
        
        # Check Debounce / Cooldown
        if (current_time - last_time) < self.COOLDOWN_SECONDS:
            return 
            
        # Update cooldown timestamp immediately
        self.last_alert_times[species] = current_time
        severity = self.SEVERITY_MAP.get(species, 'WARNING')
        
        # Dispatch HTTP request in a separate thread
        threading.Thread(
            target=self._send_payload_async, 
            args=(species, confidence, severity, frame_image),
            daemon=True
        ).start()

    def _send_payload_async(self, species, confidence, severity, frame_image):
        try:
            # Convert the full frame to base64
            _, buffer = cv2.imencode('.jpg', frame_image)
            img_b64 = base64.b64encode(buffer).decode('utf-8')
            
            payload = {
              "camera_id": self.camera_id,
              "zone": self.zone,
              "location": self.location,
              "species": species,
              "confidence": round(float(confidence), 3),
              "severity": severity,
              "timestamp": datetime.utcnow().isoformat() + "Z",
              "image_base64": img_b64 
            }
            
            response = requests.post(self.API_URL, json=payload, timeout=10)
            print(f"[ALERT SENT] {species} - Status: {response.status_code}")
        except Exception as e:
            print(f"[ALERT FAILED] {e}")

In [42]:

def process_video(video_path, alert_manager, det_conf_threshold=0.5, cls_conf_threshold=0.3, frame_skip=4):
    
    # Assuming detection_model and classify_image_cnn are defined elsewhere in your file
    # Ensure they are imported or defined before this function!
    
    cap = cv2.VideoCapture(video_path)
    frame_id = 0
    
    print(f"Processing video: {video_path}")
    
    while cap.isOpened():
        cap.set(cv2.CAP_PROP_POS_FRAMES, frame_id)
        success, frame = cap.read()
        
        if not success:
            break

        small_frame = cv2.resize(frame, (640, 360))
        detections = detection_model.single_image_detection(small_frame)
        
        current_species = set()
        max_conf_dict = {}
        
        # We no longer need best_crops_dict

        for box, conf, cls_id in zip(detections['detections'].xyxy, 
                                     detections['detections'].confidence, 
                                     detections['detections'].class_id):
            
            if cls_id == 0 and conf > det_conf_threshold:  
                x_min, y_min, x_max, y_max = map(int, box)
                cropped_img = small_frame[y_min:y_max, x_min:x_max]
                
                if cropped_img.size == 0:
                    continue
                
                species, species_conf = classify_image_cnn(cropped_img, cls_conf_threshold)
                
                if species_conf > 0.6:
                    current_species.add(species)
                    
                    if species_conf > max_conf_dict.get(species, 0):
                        max_conf_dict[species] = species_conf

                # Draw bounding box and label directly onto small_frame
                color = (0, 255, 0) if species != "Unknown" else (0, 165, 255)
                cv2.rectangle(small_frame, (x_min, y_min), (x_max, y_max), color, 2)
                label = f"{species[:15]} {species_conf:.2f}"
                cv2.putText(small_frame, label, (x_min, y_min-8),
                            cv2.FONT_HERSHEY_SIMPLEX, 0.6, color, 2)

        cv2.imshow("Wildlife Detection", small_frame)

        # Trigger alert manager
        if current_species:
            # CHANGED: Pass the fully annotated small_frame instead of the crops
            alert_manager.process_frame_detections(current_species, max_conf_dict, small_frame)

        if cv2.waitKey(1) & 0xFF == ord('q'):
            break

        frame_id += frame_skip + 1

    cap.release()
    cv2.destroyAllWindows()
    print("Video processing completed.")


if __name__ == "__main__":
    video_path = "C:/Users/yrosh/Downloads/Elephant attacks human caught on CCTV.mp4"   
    
    my_alert_manager = EdgeAlertManager(
        camera_id="CAM_Aroli_Gate_04", 
        zone="zone_yeoor_thane",
        location={"lat": 19.148, "lng": 72.932, "area_name": "Sector 6, Near Forest Boundary"}
    )
    
    process_video(
        video_path=video_path,
        alert_manager=my_alert_manager, 
        det_conf_threshold=0.5, 
        cls_conf_threshold=0.3, 
        frame_skip=4         
    )

Processing video: C:/Users/yrosh/Downloads/Elephant attacks human caught on CCTV.mp4



0: 1280x1280 1 vehicle, 735.9ms
Speed: 20.3ms preprocess, 735.9ms inference, 1.2ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 731.4ms
Speed: 22.0ms preprocess, 731.4ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 600.3ms
Speed: 21.6ms preprocess, 600.3ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 579.3ms
Speed: 20.2ms preprocess, 579.3ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 546.0ms
Speed: 13.5ms preprocess, 546.0ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 546.6ms
Speed: 17.1ms preprocess, 546.6ms inference, 0.3ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 vehicle, 555.2ms
Speed: 19.9ms preprocess, 555.2ms inference, 0.4ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 (no detections), 554.1ms
Speed: 14.9m

0: 1280x1280 1 animal, 1 person, 635.2ms
Speed: 17.2ms preprocess, 635.2ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 633.3ms
Speed: 19.4ms preprocess, 633.3ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 626.0ms
Speed: 20.0ms preprocess, 626.0ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

[ALERT SENT] Indian_Elephant - Status: 201
0: 1280x1280 1 animal, 1 person, 662.6ms
Speed: 23.1ms preprocess, 662.6ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 person, 644.0ms
Speed: 18.2ms preprocess, 644.0ms inference, 0.5ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 637.7ms
Speed: 22.4ms preprocess, 637.7ms inference, 0.6ms postprocess per image at shape (1, 3, 1280, 1280)

0: 1280x1280 1 animal, 1 person, 640.1ms
Speed: 17.8ms preprocess, 640.1ms inference, 0.5ms postprocess per i